# Mental Stress Detection (PPG) — Engine-Compatible Training Pipeline
**Dataset:** [Mental Stress PPG](https://www.kaggle.com/datasets/chtalhaanwar/mental-stress-ppg) — PPG/PRV signal, Stroop test stimulus

**Output:** `mental_stress_ppg_svm_v1.pkl` + `mental_stress_ppg_svm_v1_config.json`

Pipeline kompatibel langsung dengan **MLInferenceEngine**:
- Model disimpan sebagai `sklearn ImbPipeline` → `predict_proba()` tersedia
- `_config.json` di-generate otomatis dari `FEATURE_SCHEMA` (SSOT)
- Sinyal: `ir` (PPG infrared raw) → 12 fitur time-domain + frequency-domain

### Alur:
```
Dataset Kaggle Mental Stress PPG
    → Load CSV per subjek/kondisi (stress / no_stress)
    → Bandpass Filter (0.5–8 Hz) → Normalisasi
    → Sliding Window (64 sampel @ ~25Hz = ~2.5 detik)
    → Feature Extraction PPG: stat + peak_freq + spectral_energy (12 fitur)
    → ImbPipeline: StandardScaler → SMOTE → SVC RBF (probability=True)
    → Simpan .pkl + _config.json
```

## STEP 0 — Install & Import

In [ ]:
!pip install scikit-learn imbalanced-learn scipy -q

import os, re, zipfile, pickle, json, warnings, datetime, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.signal import butter, filtfilt
from scipy.fft import rfft, rfftfreq
from collections import Counter

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, cross_validate, GridSearchCV, train_test_split
)
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, accuracy_score, f1_score
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print('✅ Import OK')

✅ Import OK


## STEP 1 — Mount Google Drive & Setup Path

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── SESUAIKAN PATH DI SINI ───────────────────────────────────────────────────
# Download dataset dari Kaggle dulu, lalu upload ke Google Drive
# kaggle datasets download -d chtalhaanwar/mental-stress-ppg

SOURCE_ZIP    = '/content/drive/MyDrive/mental-stress/mental-stress-ppg.zip'
EXTRACT_DIR   = '/content/Dataset'
DRIVE_OUT_DIR = '/content/drive/MyDrive/mental-stress/output/'

MODEL_FILENAME  = 'mental_stress_ppg_svm_v1.pkl'
CONFIG_FILENAME = 'mental_stress_ppg_svm_v1_config.json'

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

# Ekstrak ZIP
!cp "{SOURCE_ZIP}" /content/dataset.zip
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile('/content/dataset.zip', 'r') as z:
    z.extractall(EXTRACT_DIR)

# Ekstrak ZIP bersarang (jika ada)
changed = True
while changed:
    changed = False
    for root, dirs, files in os.walk(EXTRACT_DIR):
        for f in files:
            if f.endswith('.zip'):
                zp  = os.path.join(root, f)
                out = zp.replace('.zip', '')
                os.makedirs(out, exist_ok=True)
                with zipfile.ZipFile(zp, 'r') as z2:
                    z2.extractall(out)
                os.remove(zp)
                changed = True

# Temukan semua CSV
all_csv = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    for f in files:
        if f.lower().endswith('.csv'):
            all_csv.append(os.path.join(root, f))

print(f'✅ Ekstraksi selesai — {len(all_csv)} file CSV ditemukan')

# Preview struktur folder & nama file
print('\nContoh nama file:')
for fp in all_csv[:10]:
    print(f'  {fp}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Ekstraksi selesai — 1 file CSV ditemukan

Contoh nama file:
  /content/Dataset/data.csv


## STEP 2 — Konfigurasi

In [ ]:
# ── Sensor / window settings ─────────────────────────────────────────────────
# Dataset ini direkam ~25 Hz (sesuaikan setelah inspeksi di Step 3)
FS          = 25      # Sampling rate (Hz) — sesuaikan setelah cek data
WINDOW_SIZE = 64      # 64 sampel @ 25Hz ≈ 2.56 detik per window
STEP_SIZE   = 32      # 50% overlap

# ── Label Mapping ─────────────────────────────────────────────────────────────
# Kaggle Mental Stress PPG — folder/nama file biasanya:
#   'stress'    → kondisi stres (saat Stroop test berlangsung)
#   'no_stress' atau 'baseline' atau 'relax' → kondisi rileks
# Sesuaikan keyword setelah melihat nama file di Step 1
LABEL_MAP = [
    ('stress',    'stress'),     # semua file yang mengandung 'stress'
    ('no_stress', 'no_stress'),  # atau 'baseline', 'relax', dsb
    ('baseline',  'no_stress'),  # alias baseline → no_stress
    ('relax',     'no_stress'),  # alias relax → no_stress
    ('rest',      'no_stress'),  # alias rest → no_stress
]
# ⚠ CATATAN: 'no_stress' harus dicek SEBELUM 'stress' — lebih spesifik dulu!
# Jika nama file mengandung 'no_stress', keyword 'stress' juga akan match.
# Pastikan 'no_stress' ada di urutan atas.

# ── Model Metadata ────────────────────────────────────────────────────────────
MODEL_NAME           = 'mental_stress_ppg_svm_v1'
MODEL_VERSION        = '1.0.0'
AUTHOR               = 'ppg-team'
CONFIDENCE_THRESHOLD = 0.40  # Threshold lebih tinggi untuk aplikasi medis

print('✅ Konfigurasi OK')
print(f'   Window : {WINDOW_SIZE} sampel = {WINDOW_SIZE/FS:.2f}s @ {FS}Hz')
print(f'   Step   : {STEP_SIZE} sampel (overlap {100*(1-STEP_SIZE/WINDOW_SIZE):.0f}%)')
print(f'   Label map: {[k for k,_ in LABEL_MAP]}')

✅ Konfigurasi OK
   Window : 64 sampel = 2.56s @ 25Hz
   Step   : 32 sampel (overlap 50%)
   Label map: ['stress', 'no_stress', 'baseline', 'relax', 'rest']


## STEP 3 — Load Dataset & Filter

In [ ]:
# ── Inspeksi satu file dulu untuk memahami format ─────────────────────────────
if all_csv:
    sample_fp = all_csv[0]
    sample_df = pd.read_csv(sample_fp, nrows=5)
    print(f'Kolom: {sample_df.columns.tolist()}')
    print(f'Shape: {sample_df.shape}')
    print(sample_df.head())
    print(f'\nNama file: {os.path.basename(sample_fp)}')

Kolom: ['subject ID', 'labels', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', 

In [ ]:
# ── Sesuaikan nama kolom PPG setelah inspeksi di atas ────────────────────────
# Kemungkinan nama kolom PPG: 'ppg', 'ir', 'signal', 'value', kolom ke-0, dsb.
# Ubah PPG_COL sesuai hasil inspeksi:
PPG_COL = None   # None = auto-detect kolom numerik pertama

def get_label(filepath):
    """Deteksi label dari nama file/folder."""
    path_lower = filepath.lower().replace('\\', '/')
    # Cek keyword lebih spesifik dulu (no_stress sebelum stress)
    ordered = sorted(LABEL_MAP, key=lambda x: -len(x[0]))
    for keyword, label in ordered:
        if keyword in path_lower:
            return label
    return None

def bandpass_filter(sig, lowcut=0.5, highcut=8.0, fs=FS, order=3):
    """Bandpass filter untuk PPG: hilangkan noise DC dan high-freq."""
    nyq  = 0.5 * fs
    low  = lowcut  / nyq
    high = highcut / nyq
    high = min(high, 0.99)  # Pastikan tidak >= 1.0
    b, a = butter(order, [low, high], btype='band')
    if len(sig) < 3 * max(len(b), len(a)):
        return sig  # Sinyal terlalu pendek untuk filter
    return filtfilt(b, a, sig)

def normalize_signal(sig):
    """Min-max normalisasi per window."""
    mn, mx = sig.min(), sig.max()
    if mx - mn < 1e-8:
        return np.zeros_like(sig)
    return (sig - mn) / (mx - mn)

def read_ppg_signal(fp):
    """Load sinyal PPG dari CSV."""
    try:
        df = pd.read_csv(fp, header=None)   # coba tanpa header dulu
        # Jika baris pertama bukan angka, anggap header
        try:
            float(df.iloc[0, 0])
        except (ValueError, TypeError):
            df = pd.read_csv(fp)

        if PPG_COL is not None and PPG_COL in df.columns:
            sig = df[PPG_COL].values.astype(np.float32)
        else:
            # Auto-detect: ambil kolom numerik pertama
            num_cols = df.select_dtypes(include=[np.number]).columns
            if len(num_cols) == 0:
                return None
            # Jika ada banyak kolom, ambil kolom pertama
            sig = df[num_cols[0]].values.astype(np.float32)

        sig = sig[np.isfinite(sig)]  # Hapus NaN/Inf
        if len(sig) < WINDOW_SIZE:
            return None
        return sig
    except Exception as e:
        return None

# ── Load semua file ───────────────────────────────────────────────────────────
records, n_skipped, n_short = [], 0, 0
label_counts = Counter()

for fp in all_csv:
    lbl = get_label(fp)
    if lbl is None:
        n_skipped += 1
        continue

    sig = read_ppg_signal(fp)
    if sig is None:
        n_short += 1
        continue

    # Bandpass filter → normalisasi
    sig_filt = bandpass_filter(sig.astype(np.float64), fs=FS)
    sig_norm = normalize_signal(sig_filt)

    records.append((sig_norm.astype(np.float32), lbl))
    label_counts[lbl] += 1

print(f'✅ Berhasil dimuat : {len(records)} file')
print(f'   Tidak berlabel  : {n_skipped}')
print(f'   Terlalu pendek  : {n_short}')
print(f'\nDistribusi file:')
for k, v in sorted(label_counts.items()):
    print(f'   {k}: {v} file')

✅ Berhasil dimuat : 0 file
   Tidak berlabel  : 1
   Terlalu pendek  : 0

Distribusi file:


## STEP 4 — FEATURE_SCHEMA + extract_features() + Sliding Window

In [ ]:
# ── FEATURE_SCHEMA (SSOT) ─────────────────────────────────────────────────────
# Sinyal PPG → sinyal 'ir' di engine
# Fitur: time-domain stat + frequency-domain (peak_freq, spectral_energy)

FEATURE_SCHEMA = []

# Time-domain features
for stat, cfg_stat in [
    ('mean',   'mean'),
    ('std',    'std'),
    ('min',    'min'),
    ('max',    'max'),
    ('rms',    'rms'),
    ('range',  'range'),
    ('skew',   'skew'),
    ('kurt',   'kurt'),
    ('median', 'median'),
    ('q1',     'p25'),
    ('q3',     'p75'),
    ('zcr',    'zcr'),
]:
    FEATURE_SCHEMA.append((
        f'ir_{stat}',
        {"name": f'ir_{stat}', "type": "stat", "signal": "ir",
         "stat": cfg_stat, "default": 0.0}
    ))

# Frequency-domain features
FEATURE_SCHEMA.append((
    'ir_peak_freq',
    {"name": "ir_peak_freq", "type": "stat", "signal": "ir",
     "stat": "peak_freq", "default": 0.0}
))
FEATURE_SCHEMA.append((
    'ir_spectral_energy',
    {"name": "ir_spectral_energy", "type": "stat", "signal": "ir",
     "stat": "spectral_energy", "default": 0.0}
))

N_FEATURES = len(FEATURE_SCHEMA)
print(f'Total fitur: {N_FEATURES}')
for i, (name, _) in enumerate(FEATURE_SCHEMA):
    print(f'  [{i:>3}] {name}')

Total fitur: 14
  [  0] ir_mean
  [  1] ir_std
  [  2] ir_min
  [  3] ir_max
  [  4] ir_rms
  [  5] ir_range
  [  6] ir_skew
  [  7] ir_kurt
  [  8] ir_median
  [  9] ir_q1
  [ 10] ir_q3
  [ 11] ir_zcr
  [ 12] ir_peak_freq
  [ 13] ir_spectral_energy


In [ ]:
def peak_freq(sig, fs=FS):
    """Frekuensi dominan dari FFT (Hz)."""
    n    = len(sig)
    spec = np.abs(rfft(sig))
    freq = rfftfreq(n, d=1.0/fs)
    # Skip DC (index 0)
    idx  = np.argmax(spec[1:]) + 1
    return float(freq[idx])

def spectral_energy(sig):
    """Energi spektral ternormalisasi."""
    return float(np.sum(np.abs(rfft(sig))**2) / len(sig))

def extract_features(win):
    """
    Ekstrak 14 fitur dari satu window PPG (1D).
    win: ndarray shape (WINDOW_SIZE,) — sinyal ir
    return: ndarray shape (N_FEATURES,) dtype float64
    URUTAN HARUS IDENTIK DENGAN FEATURE_SCHEMA.
    """
    ir = win.astype(np.float64)
    feats = [
        # Time-domain
        np.mean(ir),
        np.std(ir),
        np.min(ir),
        np.max(ir),
        float(np.sqrt(np.mean(ir**2))),                    # rms
        float(np.max(ir) - np.min(ir)),                    # range
        float(stats.skew(ir)),                             # skewness
        float(stats.kurtosis(ir)),                         # kurtosis (Fisher)
        float(np.median(ir)),                              # median
        float(np.percentile(ir, 25)),                      # q1
        float(np.percentile(ir, 75)),                      # q3
        float(np.sum(np.diff(np.sign(ir)) != 0) / len(ir)), # zcr
        # Frequency-domain
        peak_freq(ir, fs=FS),
        spectral_energy(ir),
    ]
    return np.array(feats, dtype=np.float64)

# ── Sliding Window ─────────────────────────────────────────────────────────────
X_list, y_list = [], []

for sig, lbl in records:
    data = sig.astype(np.float32)
    for start in range(0, len(data) - WINDOW_SIZE + 1, STEP_SIZE):
        win = data[start : start + WINDOW_SIZE]
        X_list.append(extract_features(win))
        y_list.append(lbl)

X     = np.array(X_list, dtype=np.float64)
y_raw = np.array(y_list)

# Encode label (alphabetical: no_stress=0, stress=1)
le      = LabelEncoder()
y       = le.fit_transform(y_raw)
CLASSES = list(le.classes_)

# Bersihkan NaN/Inf
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

win_dist = Counter(y_raw)
print(f'✅ X shape: {X.shape} | dtype: {X.dtype}')
print(f'   Kelas   : {CLASSES}')
print(f'   Distribusi window:')
for lbl, cnt in sorted(win_dist.items()):
    pct = 100*cnt/len(y_raw)
    print(f'     {lbl}: {cnt} windows ({pct:.1f}%)')
ratio = max(win_dist.values()) / min(win_dist.values())
print(f'   Rasio imbalance: {ratio:.1f}x')
print(f'   → Strategi: SMOTE (di dalam pipeline) + class_weight="balanced"')

✅ X shape: (0,) | dtype: float64
   Kelas   : []
   Distribusi window:


ValueError: max() iterable argument is empty

## STEP 5 — Visualisasi Distribusi & Sanity Check Sinyal

In [ ]:
labels_s = sorted(win_dist.keys())
counts_s = [win_dist[l] for l in labels_s]
colors   = ['#1565C0', '#E65100'][:len(labels_s)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Distribusi window
bars = axes[0].bar(labels_s, counts_s, color=colors, edgecolor='white', alpha=0.85)
axes[0].bar_label(bars, padding=3, fontsize=10)
axes[0].set_title('Distribusi Window per Label', fontweight='bold')
axes[0].set_ylabel('Jumlah Window')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Pie chart
axes[1].pie(counts_s, labels=labels_s, autopct='%1.1f%%', colors=colors,
            startangle=90, wedgeprops={'edgecolor':'white'})
axes[1].set_title('Proporsi Label', fontweight='bold')

# Plot 3: Contoh sinyal PPG per kelas
t = np.arange(WINDOW_SIZE) / FS
for i, lbl in enumerate(labels_s):
    # Ambil satu window contoh
    idx = np.where(y_raw == lbl)[0][0]
    # Cari record aslinya
    for rec_sig, rec_lbl in records:
        if rec_lbl == lbl and len(rec_sig) >= WINDOW_SIZE:
            sample_win = rec_sig[:WINDOW_SIZE]
            axes[2].plot(t, sample_win, label=lbl, color=colors[i], alpha=0.8)
            break
axes[2].set_title('Contoh Sinyal PPG per Kelas', fontweight='bold')
axes[2].set_xlabel('Waktu (s)')
axes[2].set_ylabel('Amplitudo (ternormalisasi)')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/distribusi_label.png', dpi=130)
plt.show()
print('✅ Visualisasi disimpan: /content/distribusi_label.png')

In [ ]:
# Visualisasi distribusi fitur per kelas
fig, axes = plt.subplots(2, 7, figsize=(20, 6))
axes = axes.flatten()
feat_names = [name for name, _ in FEATURE_SCHEMA]

for i, fname in enumerate(feat_names):
    for j, lbl in enumerate(labels_s):
        mask = y_raw == lbl
        axes[i].hist(X[mask, i], bins=30, alpha=0.6, label=lbl, color=colors[j], density=True)
    axes[i].set_title(fname, fontsize=8, fontweight='bold')
    axes[i].tick_params(labelsize=7)
    axes[i].grid(alpha=0.2)

axes[0].legend(fontsize=7)
plt.suptitle('Distribusi Fitur per Kelas (PPG Mental Stress)', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/feature_distribution.png', dpi=130, bbox_inches='tight')
plt.show()

## STEP 6 — Train/Test Split (Stratified 80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]} windows')
print(f'Test  : {X_test.shape[0]} windows')
print()
for i, cls in enumerate(CLASSES):
    n_tr = np.sum(y_train == i)
    n_te = np.sum(y_test  == i)
    print(f'  {cls}: train={n_tr}, test={n_te}')

## STEP 7 — Training Pipeline (Baseline CV + GridSearchCV)

In [ ]:
# ── Baseline CV dulu — cek apakah pipeline berjalan dengan baik ────────────
# k_neighbors SMOTE disesuaikan: min(3, min_class_count - 1)
min_class = min(Counter(y_train).values())
k_smote   = min(3, min_class - 1)
print(f'k_neighbors SMOTE: {k_smote} (min kelas train: {min_class})')

baseline_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42, k_neighbors=k_smote)),
    ('svm',    SVC(kernel='rbf', C=1.0, gamma='scale',
                   class_weight='balanced', random_state=42, probability=True))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
res = cross_validate(
    baseline_pipe, X_train, y_train, cv=cv,
    scoring=['accuracy', 'f1_macro'], n_jobs=-1
)
print(f'\nBaseline CV (C=1, gamma=scale):')
print(f'  Accuracy : {res["test_accuracy"].mean():.4f} ± {res["test_accuracy"].std():.4f}')
print(f'  F1 Macro : {res["test_f1_macro"].mean():.4f} ± {res["test_f1_macro"].std():.4f}')

In [ ]:
# ── GridSearchCV ────────────────────────────────────────────────────────────
# SVM Binary: C dan gamma paling berpengaruh
param_grid = {
    'svm__C'    : [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.01, 0.001],
}

tune_pipe = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote',  SMOTE(random_state=42, k_neighbors=k_smote)),
    ('svm',    SVC(kernel='rbf', class_weight='balanced',
                   random_state=42, probability=True))
])

grid_search = GridSearchCV(
    tune_pipe, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1_macro',   # Cocok untuk binary stress detection
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'\nBest params : {grid_search.best_params_}')
print(f'Best F1 CV  : {grid_search.best_score_:.4f}')
best_model = grid_search.best_estimator_

## STEP 8 — Evaluasi Test Set + Confusion Matrix

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)
acc    = accuracy_score(y_test, y_pred)
f1m    = f1_score(y_test, y_pred, average='macro')
f1w    = f1_score(y_test, y_pred, average='weighted')

print('── Classification Report ──')
print(classification_report(y_test, y_pred, target_names=CLASSES, digits=4))
print(f'Accuracy    : {acc:.4f} ({acc*100:.2f}%)')
print(f'F1 Macro    : {f1m:.4f}')
print(f'F1 Weighted : {f1w:.4f}')

# AUC (binary → pakai predict_proba[:,1])
if len(CLASSES) == 2:
    stress_idx = CLASSES.index('stress') if 'stress' in CLASSES else 1
    auc = roc_auc_score(y_test, y_prob[:, stress_idx])
    print(f'AUC-ROC     : {auc:.4f}')
else:
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')
    print(f'AUC OvR     : {auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix count
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred),
    display_labels=CLASSES
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Count)', fontweight='bold')

# Confusion matrix normalized
ConfusionMatrixDisplay(
    np.round(confusion_matrix(y_test, y_pred, normalize='true'), 3),
    display_labels=CLASSES
).plot(ax=axes[1], colorbar=False, cmap='Greens')
axes[1].set_title('Confusion Matrix (Normalized)', fontweight='bold')

plt.suptitle(f'Mental Stress PPG — SVM RBF | Acc={acc:.3f} F1={f1m:.3f}',
             fontweight='bold')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=130)
plt.show()
print('✅ Confusion matrix disimpan')

## STEP 9 — Verifikasi Engine Compatibility ⚠ JANGAN SKIP

In [ ]:
print('── Verifikasi Engine Compatibility ──')

# 1. predict_proba ada?
assert hasattr(best_model, 'predict_proba'), '✗ predict_proba tidak ada!'
print('✓ predict_proba() tersedia')

# 2. Output shape benar?
X_dummy = np.random.randn(5, N_FEATURES).astype(np.float64)
proba   = best_model.predict_proba(X_dummy)
assert proba.shape == (5, len(CLASSES)), f'✗ Shape salah: {proba.shape}'
print(f'✓ Output shape: {proba.shape} — OK')

# 3. Probabilitas sum ke 1?
assert np.allclose(proba.sum(axis=1), 1.0, atol=1e-5), '✗ Proba tidak sum ke 1!'
print(f'✓ Proba sum ≈ 1.0 (max_err={np.abs(proba.sum(axis=1)-1.0).max():.2e})')

# 4. Urutan kelas konsisten?
clf_step    = best_model.steps[-1][1]   # SVC
clf_classes = list(clf_step.classes_)
assert clf_classes == CLASSES, f'✗ Mismatch: clf={clf_classes}, CLASSES={CLASSES}'
print(f'✓ Urutan kelas konsisten: {CLASSES}')

# 5. Jumlah fitur = FEATURE_SCHEMA?
assert N_FEATURES == len(FEATURE_SCHEMA), f'✗ Mismatch: {N_FEATURES} vs {len(FEATURE_SCHEMA)}'
print(f'✓ Jumlah fitur: {N_FEATURES}')

# 6. probability=True di SVC?
assert clf_step.probability == True, '✗ SVC probability=False!'
print(f'✓ SVC probability=True')

print('\n✅ Semua cek passed — model siap disimpan')

## STEP 10 — Simpan .pkl + Auto-generate _config.json

In [ ]:
MODEL_PATH  = f'/content/{MODEL_FILENAME}'
CONFIG_PATH = f'/content/{CONFIG_FILENAME}'

# A. Simpan model — LANGSUNG ImbPipeline (bukan dict)
pickle.dump(best_model, open(MODEL_PATH, 'wb'), protocol=4)
print(f'✅ Model : {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1024:.1f} KB)')

# B. Generate config JSON dari FEATURE_SCHEMA (SSOT)
config = {
    "model_name"    : MODEL_NAME,
    "model_version" : MODEL_VERSION,
    "description"   : (
        f"Mental stress binary classification from PPG signal. "
        f"2 kelas: {CLASSES}. "
        f"Dataset: Kaggle Mental Stress PPG (Stroop test). "
        f"SVM RBF + StandardScaler + SMOTE."
    ),
    "author"        : AUTHOR,
    "trained_at"    : datetime.datetime.now().strftime('%Y-%m'),
    "labels"        : CLASSES,                              # dari LabelEncoder — alphabetical
    "features"      : [cfg for _, cfg in FEATURE_SCHEMA],  # dari SSOT
    "skip_if"       : {
        "finger_required": True,  # Hanya proses jika jari terdeteksi
        "require_signals": ["ir"] # Skip jika sinyal ir tidak tersedia
    },
    "output"        : {"confidence_threshold": CONFIDENCE_THRESHOLD},
    "_training_info": {           # Informatif — tidak dibaca engine
        "window_size"   : WINDOW_SIZE,
        "step_size"     : STEP_SIZE,
        "fs_hz"         : FS,
        "n_features"    : N_FEATURES,
        "n_train"       : int(X_train.shape[0]),
        "n_test"        : int(X_test.shape[0]),
        "best_params"   : grid_search.best_params_,
        "test_accuracy" : round(float(acc), 4),
        "test_f1_macro" : round(float(f1m), 4),
        "test_auc"      : round(float(auc), 4),
        "dataset_source": "https://www.kaggle.com/datasets/chtalhaanwar/mental-stress-ppg"
    }
}

with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=4, ensure_ascii=False)
print(f'✅ Config: {CONFIG_PATH}')

# Preview config (tanpa _training_info)
preview = {k: v for k, v in config.items() if k != '_training_info'}
print('\n── Config Preview ──')
print(json.dumps(preview, indent=2, ensure_ascii=False))

In [ ]:
# C. Copy semua output ke Google Drive
files_to_copy = [
    MODEL_FILENAME,
    CONFIG_FILENAME,
    'confusion_matrix.png',
    'distribusi_label.png',
    'feature_distribution.png',
]

for fname in files_to_copy:
    src = f'/content/{fname}'
    if os.path.exists(src):
        dst = os.path.join(DRIVE_OUT_DIR, fname)
        shutil.copy(src, dst)
        print(f'✅ Copied: {fname} → {dst}')
    else:
        print(f'⚠ Tidak ditemukan: {fname}')

## STEP 11 — Smoke Test (Simulasi Inferensi Engine)

In [ ]:
# Load ulang dari file (pastikan serialisasi benar)
model_loaded = pickle.load(open(MODEL_PATH, 'rb'))

X_sim  = X_test[:5].astype(np.float64)
proba  = model_loaded.predict_proba(X_sim)
labels = model_loaded.predict(X_sim)

print(f'Input  shape : {X_sim.shape}')
print(f'Output shape : {proba.shape}')
print()
print('── Simulasi Output Engine ──')
for i in range(len(X_sim)):
    pred_label  = CLASSES[labels[i]]
    confidence  = proba[i].max()
    proba_str   = ' | '.join([f'{CLASSES[j]}:{proba[i,j]:.3f}' for j in range(len(CLASSES))])
    uncertain   = '⚠ uncertain' if confidence < CONFIDENCE_THRESHOLD else ''
    true_label  = CLASSES[y_test[i]]
    correct     = '✓' if pred_label == true_label else '✗'
    print(f'  Sample {i}: {correct} pred={pred_label} ({confidence:.1%}) true={true_label} {uncertain}')
    print(f'          Proba: {proba_str}')

print(f'\nclasses_       : {list(model_loaded.steps[-1][1].classes_)}')
print(f'n_features_in_ : {getattr(model_loaded, "n_features_in_", "N/A")}')
print('\n✅ Smoke test selesai')

## STEP 12 — Laporan Akhir

In [ ]:
print('='*60)
print('           LAPORAN MODEL FINAL')
print('='*60)
print(f'  Dataset          : Mental Stress PPG (Kaggle)')
print(f'  Sinyal           : PPG infrared (ir)')
print(f'  Classifier       : SVM RBF')
print(f'  Best params      : {grid_search.best_params_}')
print(f'  Window size      : {WINDOW_SIZE} sampel ({WINDOW_SIZE/FS:.2f}s @ {FS}Hz)')
print(f'  Jumlah fitur     : {N_FEATURES}')
print(f'  Kelas            : {CLASSES}')
print()
print(f'  ── Test Set ──')
print(f'  Accuracy         : {acc:.4f} ({acc*100:.2f}%)')
print(f'  F1 Macro         : {f1m:.4f}')
print(f'  F1 Weighted      : {f1w:.4f}')
print(f'  AUC-ROC          : {auc:.4f}')
print()
print(f'  ── Engine Compatibility ──')
print(f'  Format           : ImbPipeline (sklearn-compatible)')
print(f'  Output model     : {MODEL_FILENAME}')
print(f'  Output config    : {CONFIG_FILENAME}')
print(f'  Deploy ke folder : models/ppg/')
print(f'  Confidence thr   : {CONFIDENCE_THRESHOLD} (return "uncertain" jika di bawah ini)')
print()
print(f'  ── File Output ──')
print(f'  {DRIVE_OUT_DIR}{MODEL_FILENAME}')
print(f'  {DRIVE_OUT_DIR}{CONFIG_FILENAME}')
print('='*60)